In [68]:
#Importing packages and setting parameters for initial analysis
import geopandas as gpd
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append('../src')
from ntl_functions import plot_NASA_NTL, filter_dataset_by_bounding_box, mask_dataset_by_geometry, yearly_nonzero_pixels, yearly_radiance

ImportError: cannot import name 'yearly_nonzero_pixels' from 'ntl_functions' (c:\Users\samsa\Documents\NTL_Somaliland\notebooks\../src\ntl_functions.py)

In [ ]:
SL_berbera = xr.open_dataset("../data/NTL_Data_A4/141225_SL_Berbera_VNP46A4_EY_2012_24.nc")
somaliland_shp_berbera = gpd.read_file("../data/Combined_Datasets/Somaliland_with_berbera/somaliland_with_berbera.shp")

logged_NTL = xr.where(
    SL_berbera.isnull(), 
    np.nan,
    xr.where(
        SL_berbera > 0,
        np.log(SL_berbera),
        0
    )
)

logged_NTL = mask_dataset_by_geometry(logged_NTL, somaliland_shp_berbera)

YEARS = range(2012, 2024)
VAR = "NearNadir_Composite_Snow_Free"


#Cities
HGA = gpd.read_file("../data/Combined_Datasets/Hargeisa_shp/HGA.shp")
Berbera = gpd.read_file("../data/Combined_Datasets/Berbera_shp/Berbera.shp")
Burao = gpd.read_file("../data/Combined_Datasets/Burao_shp/Burao.shp")
Boroma = gpd.read_file("../data/Combined_Datasets/Boroma_shp/Boroma.shp")
Erigavo = gpd.read_file("../data/Combined_Datasets/Erigavo_shp/Erigavo.shp")
Las_Anod = gpd.read_file("../data/Combined_Datasets/Las_Anod_shp/Las_Anod.shp")

cities_names = ["Hargeisa",
                "Burao", 
                "Boroma", 
                "Las Anod", 
                "Erigavo",
                "Berbera"
                ]


# For graphing
HGA_NTL = filter_dataset_by_bounding_box(SL_berbera, HGA)
Berbera_NTL = filter_dataset_by_bounding_box(SL_berbera, Berbera)
Burao_NTL = filter_dataset_by_bounding_box(SL_berbera, Burao)
Boroma_NTL = filter_dataset_by_bounding_box(SL_berbera, Boroma)
Erigavo_NTL = filter_dataset_by_bounding_box(SL_berbera, Erigavo)
Las_Anod_NTL = filter_dataset_by_bounding_box(SL_berbera, Las_Anod)

cities_datasets = [HGA_NTL, 
        Burao_NTL,          
        Boroma_NTL,
        Las_Anod_NTL, 
        Erigavo_NTL,
        Berbera_NTL          
         ]

c:\Users\samsa\Documents\NTL_Somaliland\venv\Lib\site-packages\xarray\computation\apply_ufunc.py:818: RuntimeWarning: divide by zero encountered in log
  result_data = func(*input_data)


In [57]:
#Showing the total number of pixels in Somaliland, the number of pixels with light detected each year, and the total radiance each year. 
non_zero_pixels = []
radiance_data = []
for year in range(2012, 2024):
    non_zero_pixels.append(round((SL_berbera["NearNadir_Composite_Snow_Free"].sel(time=f"{year}-01-01") > 0).sum().item(), 1))
    radiance_data.append(round((SL_berbera["NearNadir_Composite_Snow_Free"].sel(time=f"{year}-01-01")).sum().item(), 1))

total_rad_and_pixels = pd.DataFrame({
    "Year": list(range(2012, 2024)),
    "non_zero_pixels": non_zero_pixels,
    "radiance_data": radiance_data
})


regions_rad = {"Maroodi Jeex":[], "Togdheer":[], "Awdal":[], "Sool":[], "Sanaag":[], "Sahil":[]}
regions_pixels = {"Maroodi Jeex":[], "Togdheer":[], "Awdal":[], "Sool":[], "Sanaag":[], "Sahil":[]}

for region in regions_rad:
    regions_rad[region] = filter_dataset_by_bounding_box(mask_dataset_by_geometry(SL_berbera, somaliland_shp_berbera[somaliland_shp_berbera["admin1Name"]==region]), somaliland_shp_berbera[somaliland_shp_berbera["admin1Name"]== region])
    regions_pixels[region] = filter_dataset_by_bounding_box(mask_dataset_by_geometry(SL_berbera, somaliland_shp_berbera[somaliland_shp_berbera["admin1Name"]==region]), somaliland_shp_berbera[somaliland_shp_berbera["admin1Name"]== region])


all_region_yearly_sums = {}
for i in regions_rad:
    region_data = regions_rad[i]
    yearly_sums = []
    for year in range(2012, 2024):
        value = region_data["NearNadir_Composite_Snow_Free"].sel(time=f"{year}-01-01").sum().item()
        yearly_sums.append(round(value, 1))
    all_region_yearly_sums[i] = yearly_sums

region_rad_df = pd.DataFrame(all_region_yearly_sums)
region_rad_df.insert(0, "Year", range(2012, 2024))
region_names = list(regions_rad.keys())
region_rad_df["Total"] = region_rad_df[region_names].sum(axis=1)


all_region_yearly_pixels = {}
for i in regions_pixels:
    region_data = regions_pixels[i]
    yearly_pixels = []
    for year in range(2012, 2024):
        yearly_pixels.append((region_data["NearNadir_Composite_Snow_Free"].sel(time=f"{year}-01-01") > 0).sum().item())
    all_region_yearly_pixels[i] = yearly_pixels

region_pixel_df = pd.DataFrame(all_region_yearly_pixels)
region_pixel_df.insert(0, "Year", range(2012, 2024))
region_pixel_df["Total"] = region_pixel_df[region_names].sum(axis=1)

all_city_yearly_sums = {}
for city_name, city_data in zip(cities_names, cities_datasets):
    yearly_sums = []
    for year in range(2012, 2024):
        value = city_data["NearNadir_Composite_Snow_Free"].sel(time=f"{year}-01-01").sum().item()
        yearly_sums.append(round(value, 1))
    all_city_yearly_sums[city_name] = yearly_sums

cities_rad = pd.DataFrame(all_city_yearly_sums)
cities_rad.insert(0, "Year", range(2012, 2024))
cities_rad["Total Cities"] = cities_rad[cities_names].sum(axis=1)

all_city_yearly_pixels = {}
for city_name, city_data in zip(cities_names, cities_datasets):
    yearly_pixels = []
    for year in range(2012, 2024):
        value = (city_data["NearNadir_Composite_Snow_Free"].sel(time=f"{year}-01-01") > 0).sum().item()
        yearly_pixels.append(round(value, 1))
    all_city_yearly_pixels[city_name] = yearly_pixels

cities_pixel_df = pd.DataFrame(all_city_yearly_pixels)
cities_pixel_df.insert(0, "Year", range(2012, 2024))
cities_pixel_df["Total Cities"] = cities_pixel_df[cities_names].sum(axis=1)

In [64]:
cities_rad
#cities_pixel_df

,Year,Hargeisa,Burao,Boroma,Las Anod,Erigavo,Berbera,Total Cities
0,2012,1193.6,161.4,79.9,11.8,10.8,243.4,1700.9
1,2013,1168.9,162.4,108.4,51.6,10.1,262.6,1764.0
2,2014,1306.9,179.7,100.7,70.9,14.4,248.6,1921.2
3,2015,1483.6,179.7,97.6,70.3,14.2,266.8,2112.2
4,2016,1576.3,197.5,93.9,86.5,17.0,284.3,2255.5
5,2017,1843.6,243.3,121.5,86.7,15.2,314.1,2624.4
6,2018,2272.8,304.6,164.9,90.6,22.9,495.9,3351.7
7,2019,2696.4,436.7,213.9,106.1,40.4,586.8,4080.3
8,2020,3416.8,528.2,310.8,142.4,58.0,777.1,5233.3
9,2021,4312.2,676.7,469.6,179.6,98.6,1348.3,7085.0


In [65]:
region_rad_df
#region_pixel_df

,Year,Maroodi Jeex,Togdheer,Awdal,Sool,Sanaag,Sahil,Total
0,2012,1212.9,163.4,79.9,11.8,12.6,244.1,1724.7
1,2013,1190.3,164.4,108.4,51.6,10.1,262.6,1787.4
2,2014,1336.7,182.1,100.7,70.9,15.5,250.2,1956.1
3,2015,1530.6,182.4,97.6,70.3,14.2,270.4,2165.5
4,2016,1643.0,200.4,93.9,86.5,18.7,286.1,2328.6
5,2017,1930.7,250.5,121.5,86.7,33.0,318.4,2740.8
6,2018,2393.0,326.0,164.9,90.6,40.4,501.5,3516.4
7,2019,2874.7,473.1,213.9,107.1,60.4,593.5,4322.7
8,2020,3722.0,587.9,313.1,146.6,83.8,788.0,5641.4
9,2021,4785.2,768.6,478.8,206.6,188.7,1372.5,7800.4


In [40]:
total_rad_and_pixels

,Year,non_zero_pixels,radiance_data
0,2012,569,1724.6
1,2013,581,1787.3
2,2014,668,1956.1
3,2015,757,2165.4
4,2016,760,2328.6
5,2017,887,2740.7
6,2018,1050,3516.3
7,2019,1139,4322.6
8,2020,1318,5641.5
9,2021,1760,7800.5
